# Inference + Grad-CAM XAI Demo

Runs the full pipeline on a sample image: **dog-or-not gate -> emotion classifier -> Grad-CAM++ explanation**.

This notebook is a thin demo layer over `src/pipeline.py` -- it does not reimplement the
preprocessing/model logic (that lives in `src/`, single source of truth, shared with training).
Edit `IMAGE_PATH` below and run all cells.

In [ ]:
import sys
from pathlib import Path

# Repo root (parent of notebooks/) so `import src...` works when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src.pipeline import predict
import matplotlib.pyplot as plt


In [ ]:
# EDIT THIS -- path to any image you want to test, relative to repo root
IMAGE_PATH = Path.cwd().parent / "examples" / "sample_images" / "test_dog.jpg"

assert IMAGE_PATH.exists(), f"Image not found: {IMAGE_PATH}"
plt.imshow(plt.imread(IMAGE_PATH))
plt.axis("off")
plt.title("Input image")
plt.show()


## Run the full pipeline

`predict()` runs the dog-or-not gate first. If the image isn't a dog, it stops there --
no point running an emotion model on a non-dog image.

In [ ]:
result = predict(IMAGE_PATH)

if not result["is_dog"]:
    print(f"Not a dog -- predicted '{result['label']}' ({result['confidence']:.1f}% confidence)")
else:
    print(f"Dog detected ({result['confidence']:.1f}% confidence)")
    print(f"Emotion: {result['emotion']} ({result['confidence']:.1f}% confidence)")
    print()
    print("Class probabilities:")
    for label, pct in sorted(result["probs"].items(), key=lambda x: -x[1]):
        print(f"  {label}: {pct:.2f}%")


## Grad-CAM++ visualization

Where in the image did the model actually look when making this prediction?

In [ ]:
if result["is_dog"]:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(result["gradcam_visualization"])
    axes[0].set_title(f"Grad-CAM++: {result['emotion']}")
    axes[0].axis("off")

    im = axes[1].imshow(result["attention_grid_pct"], cmap="Blues")
    axes[1].set_title("Attention by region (%)")
    for i in range(3):
        for j in range(3):
            axes[1].text(j, i, f"{result['attention_grid_pct'][i, j]:.0f}%",
                         ha="center", va="center")
    axes[1].set_xticks([]); axes[1].set_yticks([])
    plt.tight_layout()
    plt.show()

    print(result["explanation"])
else:
    print("Skipping Grad-CAM -- no dog detected in this image.")


## Batch demo (optional)

Run the pipeline over every image in `examples/sample_images/` to sanity-check a few at once.
Set `explain=False` here since we only want labels, not Grad-CAM, for a quick batch pass.

In [ ]:
sample_dir = Path.cwd().parent / "examples" / "sample_images"

for img_path in sorted(sample_dir.glob("*")):
    if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
        continue
    r = predict(img_path, explain=False)
    if r["is_dog"]:
        print(f"{img_path.name}: dog, {r['emotion']} ({r['confidence']:.1f}%)")
    else:
        print(f"{img_path.name}: not a dog ({r['label']}, {r['confidence']:.1f}%)")
